In [1]:
import langchain

In [2]:
print(langchain.__version__)

1.2.10


In [3]:
# pip install langchain-classic


In [4]:
# Modern import for v1.x
from langchain_classic.retrievers.multi_query import MultiQueryRetriever


/Users/kristalshrestha/Documents/Code/LLM_Scratch/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("Hugging_face_api_token")
hf_embeddings_api = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",  # lightweight
    # model="Qwen/Qwen3-Embedding-8B",
    # task="feature-extraction",# heavyweight but uses a lot of quota token
    huggingfacehub_api_token=api_key,
)

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [7]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

# create vector store along with embeddings

In [8]:
vectorstore=Chroma.from_documents(
    documents=all_docs,
    embedding=hf_embeddings_api
)

# also need llm model for the multiquery retriever

In [9]:
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFacePipeline,
    HuggingFaceEndpoint,
)

from dotenv import load_dotenv
import os

# hugging face free api endpoint is unreliable so trying locally
load_dotenv()
api_key = os.getenv("Hugging_face_api_token")

llm = HuggingFaceEndpoint(
    # repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    # repo_id="google/gemma-2-2b-it",
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=api_key,
)
# os.environ["HF_HOME"] = "/Users/kristalshrestha/Documents/Code/LLM_Scratch/models"
# define the model
# llm = HuggingFacePipeline.from_model_id(
#     # this tinyllama is vert small for structured output tasks
#     # model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
#     model_id="google/gemma-2-2b-it",
#     task="text-generation",
#     pipeline_kwargs={
#         "max_new_tokens": 100,
#         "temperature": 0.5,
#     },
# )
model = ChatHuggingFace(llm=llm)

# create retrievers and comparing

In [10]:
similarity_retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

In [11]:
multiquery_retriever=MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k":5}),
    llm=model
)

In [12]:
#Query
query="How to improve energy levels and maintain balance?"

In [13]:
#Retrieve results
similarity_results=similarity_retriever.invoke(query)
multiquery_results=multiquery_retriever.invoke(query)

In [14]:
for i,doc in enumerate(similarity_results):
    print(f"\n---Result {i+1}---")
    print(doc.page_content)


---Result 1---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

---Result 2---
The solar energy system in modern homes helps balance electricity demand.

---Result 3---
Consuming leafy greens and fruits helps detox the body and improve longevity.

---Result 4---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

---Result 5---
Photosynthesis enables plants to produce energy by converting sunlight.


In [15]:
for i,doc in enumerate(multiquery_results):
    print(f"\n --Result {i+1}--")
    print(doc.page_content)


 --Result 1--
Drinking sufficient water throughout the day helps maintain metabolism and energy.

 --Result 2--
Consuming leafy greens and fruits helps detox the body and improve longevity.

 --Result 3--
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

 --Result 4--
The solar energy system in modern homes helps balance electricity demand.

 --Result 5--
Regular walking boosts heart health and can reduce symptoms of depression.

 --Result 6--
Photosynthesis enables plants to produce energy by converting sunlight.
